# 2025-2026 Interação em Linguagem Natural
# Grupo 06
# 66132 Rafael Ventura
# 58225 André Fonseca

In [1]:
!pip install -q transformers==5.10.2 torch==2.12.0 torchvision==0.27.0 --upgrade

In [2]:
import torchvision.io

if not hasattr(torchvision.io, 'VideoReader'):
    class VideoReader:
        pass
    torchvision.io.VideoReader = VideoReader

In [3]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("aps/super_glue", "multirc")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
train_ds = ds['train']
val_ds = ds['validation']
test_ds = ds['test']

print(f"Train:      {len(train_ds)} samples")
print(f"Validation: {len(val_ds)} samples")
print(f"Test:       {len(test_ds)} samples")
print(f"\Columns: {train_ds.column_names}")
print(f"\nTrain example:\n{train_ds[0]}")

Train:      27243 samples
Validation: 4848 samples
Test:       9693 samples
\Columns: ['paragraph', 'question', 'answer', 'idx', 'label']

Train example:
{'paragraph': 'While this process moved along, diplomacy continued its rounds. Direct pressure on the Taliban had proved unsuccessful. As one NSC staff note put it, "Under the Taliban, Afghanistan is not so much a state sponsor of terrorism as it is a state sponsored by terrorists." In early 2000, the United States began a high-level effort to persuade Pakistan to use its influence over the Taliban. In January 2000, Assistant Secretary of State Karl Inderfurth and the State Department\'s counterterrorism coordinator, Michael Sheehan, met with General Musharraf in Islamabad, dangling before him the possibility of a presidential visit in March as a reward for Pakistani cooperation. Such a visit was coveted by Musharraf, partly as a sign of his government\'s legitimacy. He told the two envoys that he would meet with Mullah Omar and press

<>:8: SyntaxWarning: invalid escape sequence '\C'
<>:8: SyntaxWarning: invalid escape sequence '\C'
/tmp/ipykernel_7312/1325887655.py:8: SyntaxWarning: invalid escape sequence '\C'
  print(f"\Columns: {train_ds.column_names}")


In [5]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def tokenize_function(example):
    return tokenizer(
        example['paragraph'],
        example['question'] + ' ' + example['answer'],
        truncation=True,
    )

cols_to_remove = ['paragraph', 'question', 'answer', 'idx']

tokenized_train = train_ds.map(tokenize_function, remove_columns=cols_to_remove)
tokenized_val   = val_ds.map(tokenize_function,   remove_columns=cols_to_remove)
tokenized_test  = test_ds.map(tokenize_function,  remove_columns=cols_to_remove)

tokenized_train = tokenized_train.rename_column('label', 'labels')
tokenized_val   = tokenized_val.rename_column('label', 'labels')

tokenized_train.set_format('torch')
tokenized_val.set_format('torch')
tokenized_test.set_format('torch')

print("Tokenization concluded")
print(tokenized_train)

Tokenization concluded
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 27243
})


In [ ]:
from transformers import RobertaForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate
import numpy as np


model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)
metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir='./checkpoints',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='steps',
    eval_steps=850,
    save_strategy='steps',
    save_steps=850,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=100,
    learning_rate=1e-2,
    warmup_ratio=0.1,
)
data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss


In [ ]:
# Evaluation
train_metrics = trainer.evaluate(tokenized_train, metric_key_prefix='train')
val_metrics   = trainer.evaluate(tokenized_val,   metric_key_prefix='eval')

print(f"Train accuracy: {train_metrics['train_accuracy']:.4f}")
print(f"Val   accuracy: {val_metrics['eval_accuracy']:.4f}")


trainer.save_model('./models/roberta-multirc')
tokenizer.save_pretrained('./models/roberta-multirc')
print("Saved in ./roberta-multirc")

In [ ]:
import torch

def test(passage, question, answer):
    trainer.model.eval()
    inputs = tokenizer(
        passage,
        question + ' ' + answer,
        truncation=True,
        return_tensors='pt'
    ).to(trainer.model.device)
    with torch.no_grad():
        outputs = trainer.model(**inputs)
    prediction = outputs.logits.argmax(dim=-1).item()
    return prediction

# Exemplo de uso
example = val_ds[0]
result = test(example['paragraph'], example['question'], example['answer'])
print(f"Example of using the test function:")
print(f"  Res: {result} (real label: {example['label']})")

In [ ]:
import shutil
shutil.make_archive('roberta-multirc', 'zip', './models/roberta-multirc')
from google.colab import files
files.download('roberta-multirc.zip')